# 03 - AutoML Model Training

This notebook demonstrates:
- AutoML with Bayesian Optimization
- Training Random Forest, XGBoost, LightGBM, SVM, Neural Networks
- Model comparison and selection

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import CAMELSIndDataLoader
from src.data.preprocessor import StreamflowPreprocessor
from src.features.engineer import FeatureEngineer
from src.models.automl_trainer import AutoMLTrainer
from src.models.evaluator import ModelEvaluator
from src.utils.helpers import load_config, set_seed

set_seed(42)
%matplotlib inline

In [ ]:
# Prepare data
loader = CAMELSIndDataLoader('../data/raw')
if not loader.list_catchments():
    loader.generate_sample_data(n_catchments=5, n_days=2000)

data = loader.load_all_catchments()
engineer = FeatureEngineer()
data = engineer.transform(data)

preprocessor = StreamflowPreprocessor()
X_train, X_test, y_train, y_test = preprocessor.fit_transform(data)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Run AutoML
model_config = load_config('../configs/model_configs.yaml')
trainer = AutoMLTrainer(config=model_config)

best_model = trainer.fit(X_train, y_train, n_trials=20, cv_folds=3)

print(f'\nBest model: {trainer.best_model_name}')
print(f'Best NSE: {trainer.best_score:.4f}')
print(f'\nAll results:')
for name, result in trainer.results.items():
    if 'error' not in result:
        print(f'  {name}: NSE={result["best_score"]:.4f} (time: {result["training_time_s"]}s)')

In [ ]:
# Evaluate on test set
evaluator = ModelEvaluator()
result = evaluator.evaluate(best_model, X_test, y_test, model_name=trainer.best_model_name)

print('Test Set Performance:')
for metric, value in result['overall_metrics'].items():
    print(f'  {metric.upper()}: {value}')

In [ ]:
# Save model
trainer.save_model('../outputs/models/best_model.joblib')
print('Model saved!')